In [ ]:
!pip install -q wfdb scipy "numpy<2.0" "pandas==2.2.2" tensorflow seaborn matplotlib scikit-learn

import csv
import json
import os
import random
import shutil
import tarfile
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import wfdb
from google.colab import drive
from scipy import signal as sp_signal
from scipy.signal import savgol_filter
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from tensorflow.keras import layers, models

drive.mount("/content/drive")

# ECG Arrhythmia Classifier

Datasets: MIT-BIH, PTB-XL, LTAFDB, and NSTDB.  
ECG recordings are loaded from PhysioNet.

## Pipeline

1. Install packages and set up paths
2. Extract NSR and AFIB records from MIT-BIH
3. Extract PVC and LBBB records from MIT-BIH
4. Extract NSR, AFIB, and LBBB records from PTB-XL
5. Extract AFIB and PVC records from LTAFDB
6. Extract PVC and LBBB records from INCART
7. Resample signals to 360 Hz
8. Create patient-wise train, validation, and test splits
9. Add NSTDB noise to the training data
10. Filter and normalize each ECG segment
11. Train the dual-input CNN
12. Evaluate using macro-F1 and related metrics
13. Convert the model to float32 and dynamic-range TFLite formats

## Block 1 - Setup

In [ ]:
BASE_DIR = "/content/drive/MyDrive/ecg_data"
DATASETS_TAR_DRIVE = "/content/drive/MyDrive/DATASETS.tar.gz"
RUNS_DIR = "/content/drive/MyDrive/ecg_runs"

SPLIT_DIR = os.path.join(BASE_DIR, "split")
AUG_DIR = os.path.join(BASE_DIR, "split-augmented")
FILTERED_DIR = os.path.join(BASE_DIR, "split-filtered")
SEGS_TAR_DRIVE = os.path.join(BASE_DIR, "segs.tar.gz")
AUG_TAR_DRIVE = os.path.join(BASE_DIR, "aug_segs.tar.gz")

TMP_BASE = "/tmp/ecg"
TMP_RAW_DIR = os.path.join(TMP_BASE, "raw")
NSR_AFIB_DIR = os.path.join(TMP_BASE, "segs", "nsr-afib")
PVC_LBBB_DIR = os.path.join(TMP_BASE, "segs", "pvc-lbbb")
AUG_SEGS_DIR = os.path.join(TMP_BASE, "aug_segs")

MITDB_LOCAL = os.path.join(TMP_RAW_DIR, "mitdb")
PTBXL_LOCAL = os.path.join(TMP_RAW_DIR, "ptb-xl")
LTAFDB_LOCAL = os.path.join(TMP_RAW_DIR, "ltafdb")
INCART_LOCAL = os.path.join(TMP_RAW_DIR, "incartdb")

WIN_LEN = 1024
HALF_WIN = WIN_LEN // 2
SAMPLE_RATE_HZ = 360.0
CLASSES = ["NSR", "AFIB", "PVC", "LBBB"]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASSES)}
N_WORKERS = 2

directories = [
    BASE_DIR,
    TMP_RAW_DIR,
    NSR_AFIB_DIR,
    PVC_LBBB_DIR,
    os.path.join(NSR_AFIB_DIR, "NSR"),
    os.path.join(NSR_AFIB_DIR, "AFIB"),
    os.path.join(PVC_LBBB_DIR, "PVC"),
    os.path.join(PVC_LBBB_DIR, "LBBB"),
    AUG_SEGS_DIR,
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)

In [ ]:
PROGRESS_FILE = os.path.join(BASE_DIR, "progress.json")


def _load_progress():
    try:
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def _is_done(step):
    return _load_progress().get(step, False)


def _mark_done(step):
    progress = _load_progress()
    progress[step] = True

    for attempt in range(5):
        try:
            with open(PROGRESS_FILE, "w") as f:
                json.dump(progress, f, indent=2)
                f.flush()
                os.fsync(f.fileno())
            return
        except OSError:
            if attempt < 4:
                time.sleep(10)


def _extract_tar(tar_path, dst_dir):
    os.makedirs(dst_dir, exist_ok=True)

    with tarfile.open(tar_path, "r:gz") as archive:
        archive.extractall(dst_dir)


def _restore_from_drive(tar_drive, dst_dir):
    temp_tar = f"/tmp/{os.path.basename(tar_drive)}"
    shutil.copy2(tar_drive, temp_tar)
    _extract_tar(temp_tar, dst_dir)
    os.remove(temp_tar)


def _compress_to_drive(src_dir, tar_drive, arc_name=None):
    temp_tar = f"/tmp/{os.path.basename(tar_drive)}"
    arc_name = arc_name or os.path.basename(src_dir)

    with tarfile.open(temp_tar, "w:gz") as archive:
        archive.add(src_dir, arcname=arc_name)

    shutil.copy2(temp_tar, tar_drive)
    os.remove(temp_tar)

In [ ]:
_EXTRACTION_STEPS = ("block2", "block3", "block3b", "block3c", "block3d")

if all(_is_done(step) for step in _EXTRACTION_STEPS):
    nsr_dir = os.path.join(NSR_AFIB_DIR, "NSR")
    segments_ready = os.path.isdir(nsr_dir) and bool(os.listdir(nsr_dir))

    if not segments_ready and os.path.exists(SEGS_TAR_DRIVE):
        _restore_from_drive(SEGS_TAR_DRIVE, TMP_BASE)
elif not (os.path.isdir(MITDB_LOCAL) and os.listdir(MITDB_LOCAL)):
    if os.path.exists(DATASETS_TAR_DRIVE):
        _restore_from_drive(DATASETS_TAR_DRIVE, TMP_RAW_DIR)


if _is_done("block5"):
    augmented_ready = os.path.isdir(AUG_SEGS_DIR) and bool(os.listdir(AUG_SEGS_DIR))

    if not augmented_ready and os.path.exists(AUG_TAR_DRIVE):
        _restore_from_drive(AUG_TAR_DRIVE, TMP_BASE)


def _has_hea_files(path):
    try:
        return os.path.isdir(path) and any(
            file.endswith(".hea") for file in os.listdir(path)
        )
    except OSError:
        return False


def _wfdb_args(local_dir, record, remote_db):
    if _has_hea_files(local_dir):
        return os.path.join(local_dir, record), None
    return record, remote_db


MITDB_PN = MITDB_LOCAL if _has_hea_files(MITDB_LOCAL) else "mitdb"
LTAFDB_PN = LTAFDB_LOCAL if _has_hea_files(LTAFDB_LOCAL) else "ltafdb"
INCART_PN = INCART_LOCAL if _has_hea_files(INCART_LOCAL) else "incartdb"

PTBXL_RECORDS = os.path.join(PTBXL_LOCAL, "records500")
PTBXL_LOCAL_OK = os.path.isdir(PTBXL_RECORDS) and bool(
    os.listdir(PTBXL_RECORDS)
)

_existing = _load_progress()
_done_steps = [step for step, done in _existing.items() if done]

In [ ]:
if _has_hea_files(MITDB_LOCAL):
    mitdb_records = sorted(
        os.path.splitext(file)[0]
        for file in os.listdir(MITDB_LOCAL)
        if file.endswith(".hea")
    )
else:
    mitdb_records = wfdb.get_record_list("mitdb")

print(f"MIT-BIH records: {len(mitdb_records)}")

## Block 2 - Extract NSR and AFIB Segments

MIT-BIH records are read from PhysioNet.

- Rhythm annotations define the segment boundaries.
- `(N` and `(NSR` are labeled as NSR.
- `(AFIB` is labeled as AFIB.
- Each interval is divided into non-overlapping 1,024-sample windows.
- NSR windows must contain only `N` or `.` beat annotations.
- AFIB windows containing ventricular or ectopic symbols are skipped.
- Records without an MLII channel are skipped.

Output directories:

- `nsr-afib/NSR/`
- `nsr-afib/AFIB/`

In [ ]:
def get_mlii_channel(signal_names):
    for index, name in enumerate(signal_names):
        if name.strip().upper() in {"MLII", "ML II", "II"}:
            return index
    return None


step = "block2"

if not _is_done(step):
    rhythm_map = {
        "(N": "NSR",
        "(NSR": "NSR",
        "(AFIB": "AFIB",
    }

    normal_symbols = {"N", "."}
    afib_excluded = {"V", "r", "E", "F", "L", "R", "B", "!", "[", "]"}

    for label in ("NSR", "AFIB"):
        folder = os.path.join(NSR_AFIB_DIR, label)
        shutil.rmtree(folder, ignore_errors=True)
        os.makedirs(folder, exist_ok=True)

    totals = {"NSR": 0, "AFIB": 0}

    for record in mitdb_records:
        record_path, pn_dir = _wfdb_args(MITDB_LOCAL, record, "mitdb")

        try:
            header = wfdb.rdheader(record_path, pn_dir=pn_dir)
            channel = get_mlii_channel(header.sig_name)

            if channel is None:
                continue

            signal_data = wfdb.rdrecord(
                record_path,
                channels=[channel],
                pn_dir=pn_dir,
            ).p_signal[:, 0].astype(np.float32)

            annotations = wfdb.rdann(record_path, "atr", pn_dir=pn_dir)
        except Exception:
            continue

        rhythm_events = []

        for sample, symbol, aux_note in zip(
            annotations.sample,
            annotations.symbol,
            annotations.aux_note,
        ):
            if symbol == "+":
                rhythm = aux_note.strip().rstrip("\x00").strip()
                rhythm_events.append((sample, rhythm))

        if not rhythm_events:
            continue

        intervals = []

        for index, (start, rhythm) in enumerate(rhythm_events):
            end = (
                rhythm_events[index + 1][0]
                if index + 1 < len(rhythm_events)
                else len(signal_data)
            )

            label = rhythm_map.get(rhythm)
            if label:
                intervals.append((start, end, label))

        beat_annotations = list(zip(annotations.sample, annotations.symbol))
        record_counts = {"NSR": 0, "AFIB": 0}

        for start, end, label in intervals:
            start = max(0, start)
            end = min(len(signal_data), end)

            for offset in range(start, end - WIN_LEN + 1, WIN_LEN):
                segment_end = offset + WIN_LEN
                symbols = [
                    symbol
                    for sample, symbol in beat_annotations
                    if offset <= sample < segment_end
                ]

                if label == "NSR":
                    if not symbols or not all(
                        symbol in normal_symbols for symbol in symbols
                    ):
                        continue
                elif any(symbol in afib_excluded for symbol in symbols):
                    continue

                segment = signal_data[offset:segment_end]

                filename = f"{record}_{label}_{record_counts[label]:05d}.npy"
                output_path = os.path.join(
                    NSR_AFIB_DIR,
                    label,
                    filename,
                )

                np.save(output_path, segment)
                record_counts[label] += 1

        for label in totals:
            totals[label] += record_counts[label]

    _mark_done(step)

print("Block 2 complete.")

## Block 3a - Extract PVC and LBBB Segments

MIT-BIH records are loaded from PhysioNet.

- `V` annotations are labeled as PVC.
- `L` annotations are labeled as LBBB.
- PVC windows use a jittered R-peak position:
  - 40% centered at sample 512
  - 60% uniformly distributed between samples 205 and 819
- LBBB windows are centered at sample 512.
- Windows are kept only when they contain normal beats or beats from the target class.
- Windows crossing the signal boundary are skipped.

Output directories:

- `pvc-lbbb/PVC/`
- `pvc-lbbb/LBBB/`

In [ ]:
step = "block3a"

if not _is_done(step):
    label_map = {"V": "PVC", "L": "LBBB"}
    normal_beats = {"N", "."}
    allowed_beats = {
        "V": normal_beats | {"V"},
        "L": normal_beats | {"L"},
    }

    centered_ratio = 0.40
    min_position = int(0.20 * WIN_LEN)
    max_position = int(0.80 * WIN_LEN)
    rng = random.Random(42)

    def choose_target_position():
        if rng.random() < centered_ratio:
            return HALF_WIN
        return rng.randint(min_position, max_position)

    for label in label_map.values():
        folder = os.path.join(PVC_LBBB_DIR, label)
        shutil.rmtree(folder, ignore_errors=True)
        os.makedirs(folder, exist_ok=True)

    totals = {label: 0 for label in label_map.values()}

    for record in mitdb_records:
        record_path, pn_dir = _wfdb_args(MITDB_LOCAL, record, "mitdb")

        try:
            header = wfdb.rdheader(record_path, pn_dir=pn_dir)
            channel = get_mlii_channel(header.sig_name)

            if channel is None:
                continue

            signal_data = wfdb.rdrecord(
                record_path,
                channels=[channel],
                pn_dir=pn_dir,
            ).p_signal[:, 0].astype(np.float32)

            annotations = wfdb.rdann(record_path, "atr", pn_dir=pn_dir)
        except Exception:
            continue

        beats = list(zip(annotations.sample, annotations.symbol))
        record_counts = {label: 0 for label in label_map.values()}

        for sample, symbol in beats:
            if symbol not in label_map:
                continue

            label = label_map[symbol]
            target_position = (
                choose_target_position()
                if symbol == "V"
                else HALF_WIN
            )

            start = sample - target_position
            end = start + WIN_LEN

            if start < 0 or end > len(signal_data):
                continue

            window_symbols = [
                beat_symbol
                for beat_sample, beat_symbol in beats
                if start <= beat_sample < end
            ]

            if any(
                beat_symbol not in allowed_beats[symbol]
                for beat_symbol in window_symbols
            ):
                continue

            filename = f"{record}_{label}_{record_counts[label]:05d}.npy"
            output_path = os.path.join(PVC_LBBB_DIR, label, filename)

            np.save(output_path, signal_data[start:end])
            record_counts[label] += 1

        for label in totals:
            totals[label] += record_counts[label]

    _mark_done(step)

print("Block 3a complete.")

## Block 3b - Extract NSR, AFIB, and LBBB from PTB-XL

PTB-XL records are loaded from PhysioNet.

- Records must have one dominant SCP code with at least 80% confidence:
  - `NORM` -> NSR
  - `AFIB` -> AFIB
  - `CLBBB` -> LBBB
- Records are rejected if another target-class code reaches 40% confidence.
- NSR and AFIB records are checked against the PTB-XL+ 12SL statements.
- Records containing `PVC`, `VPR`, `BIGU`, `TRIGU`, or `VESC` with at least 50% confidence are rejected.
- Lead II is extracted from the 500 Hz recording.
- Signals are resampled to 360 Hz using `resample_poly(x, 18, 25)`.
- Each 10-second recording produces approximately three non-overlapping 1,024-sample windows.
- Patient IDs are included in filenames for patient-wise splitting.

Output directories:

- `nsr-afib/NSR/`
- `nsr-afib/AFIB/`
- `pvc-lbbb/LBBB/`

In [ ]:
step = "block3b"

if not _is_done(step):
    import ast
    import threading
    import urllib.request

    PTBXL_VERSION = "ptb-xl/1.0.3"
    PTBXLPLUS_VERSION = "ptb-xl-plus/1.0.1"
    PTBXL_RESAMP_UP = 18
    PTBXL_RESAMP_DOWN = 25
    PTBXL_N_WORKERS = 8
    PTBXL_LEAD_IDX = 1
    PTBXL_MIN_CONF = 80.0
    PTBXL_EXCL_CONF = 40.0

    PTBXL_SCP_MAP = {
        "NORM": "NSR",
        "AFIB": "AFIB",
        "CLBBB": "LBBB",
    }

    ectopy_codes = {"PVC", "VPR", "BIGU", "TRIGU", "VESC"}
    meta_dir = os.path.join(BASE_DIR, "ptbxl_meta")
    os.makedirs(meta_dir, exist_ok=True)

    meta_path = os.path.join(meta_dir, "ptbxl_database.csv")

    if not os.path.exists(meta_path):
        url = f"https://physionet.org/files/{PTBXL_VERSION}/ptbxl_database.csv"
        urllib.request.urlretrieve(url, meta_path)

    df_meta = pd.read_csv(meta_path, index_col="ecg_id")
    df_meta["scp_codes"] = df_meta["scp_codes"].apply(
        lambda value: ast.literal_eval(value)
        if isinstance(value, str)
        else {}
    )

    def get_ptbxl_label(codes):
        matches = {
            code: confidence
            for code, confidence in codes.items()
            if code in PTBXL_SCP_MAP and confidence >= PTBXL_MIN_CONF
        }

        if len(matches) != 1:
            return None

        selected = next(iter(matches))

        if any(
            code != selected
            and code in PTBXL_SCP_MAP
            and confidence >= PTBXL_EXCL_CONF
            for code, confidence in codes.items()
        ):
            return None

        return PTBXL_SCP_MAP[selected]

    df_meta["ptbxl_class"] = df_meta["scp_codes"].apply(get_ptbxl_label)

    local_12sl = os.path.join(
        TMP_RAW_DIR,
        "ptb-xl-plus",
        "labels",
        "12sl_statements.csv",
    )
    cached_12sl = os.path.join(meta_dir, "12sl_statements.csv")

    if os.path.exists(local_12sl):
        statements_path = local_12sl
    else:
        statements_path = cached_12sl

        if not os.path.exists(statements_path):
            url = (
                f"https://physionet.org/files/{PTBXLPLUS_VERSION}/"
                "labels/12sl_statements.csv"
            )
            urllib.request.urlretrieve(url, statements_path)

    df_12sl = pd.read_csv(statements_path, index_col="ecg_id")

    def parse_statements(value):
        if not isinstance(value, str) or value.strip() in {"", "[]"}:
            return []

        try:
            return ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []

    df_12sl["parsed"] = df_12sl["statements_ext"].apply(parse_statements)

    ectopy_ids = {
        index
        for index, row in df_12sl.iterrows()
        if any(
            code in ectopy_codes and confidence >= 50.0
            for code, confidence in row["parsed"]
        )
    }

    target_classes = {"NSR", "AFIB"}
    guarded = df_meta["ptbxl_class"].isin(target_classes)

    df_meta.loc[
        df_meta.index.isin(ectopy_ids) & guarded,
        "ptbxl_class",
    ] = None

    df_target = df_meta[df_meta["ptbxl_class"].notna()].copy()
    ptbxl_counts = {label: 0 for label in PTBXL_SCP_MAP.values()}
    errors = 0
    lock = threading.Lock()

    def process_record(ecg_id, row):
        label = row["ptbxl_class"]
        patient_id = int(row["patient_id"])
        filename = row["filename_hr"]

        subdir, record_name = filename.rsplit("/", 1)

        if PTBXL_LOCAL_OK:
            record_path = os.path.join(PTBXL_LOCAL, subdir, record_name)
            pn_dir = None
        else:
            record_path = record_name
            pn_dir = f"{PTBXL_VERSION}/{subdir}"

        output_root = (
            NSR_AFIB_DIR
            if label in {"NSR", "AFIB"}
            else PVC_LBBB_DIR
        )
        output_dir = os.path.join(output_root, label)

        try:
            record = wfdb.rdrecord(
                record_path,
                pn_dir=pn_dir,
                channels=[PTBXL_LEAD_IDX],
            )
            signal_raw = record.p_signal[:, 0].astype(np.float64)
        except Exception as error:
            return label, 0, error

        signal_360 = sp_signal.resample_poly(
            signal_raw,
            PTBXL_RESAMP_UP,
            PTBXL_RESAMP_DOWN,
        ).astype(np.float32)

        saved = 0

        for window in range(len(signal_360) // WIN_LEN):
            start = window * WIN_LEN
            segment = signal_360[start:start + WIN_LEN]

            if len(segment) != WIN_LEN:
                continue

            filename = (
                f"ptbxl_{patient_id:05d}_{ecg_id:05d}_"
                f"w{window:02d}_{label}.npy"
            )

            np.save(os.path.join(output_dir, filename), segment)
            saved += 1

        return label, saved, None

    rows = list(df_target.iterrows())

    with ThreadPoolExecutor(max_workers=PTBXL_N_WORKERS) as pool:
        futures = {
            pool.submit(process_record, ecg_id, row): ecg_id
            for ecg_id, row in rows
        }

        for future in as_completed(futures):
            label, saved, error = future.result()

            with lock:
                if error is not None:
                    errors += 1
                else:
                    ptbxl_counts[label] += saved

    _mark_done(step)

print("Block 3b complete.")

## Block 3c - Extract AFIB and PVC from LTAFDB

LTAFDB records are loaded from PhysioNet.

- Channel 0 is used.
- Signals are resampled from 128 Hz to 360 Hz with `resample_poly(x, 45, 16)`.
- AFIB windows must belong to `(AFIB` rhythm intervals.
- AFIB windows containing ventricular or ectopic beats are skipped.
- AFIB windows use non-overlapping 1,024-sample segments, with a maximum of 2,000 per record.
- PVC windows are centered on `V` beats found outside AFIB intervals.
- PVC windows may contain only `N`, `.`, or `V` beat annotations.
- PVC windows use the same R-peak jitter as Block 3:
  - 40% centered at sample 512
  - 60% uniformly positioned between samples 205 and 819
- PVC extraction is limited to 200 segments per record.

Output directories:

- `nsr-afib/AFIB/`
- `pvc-lbbb/PVC/`

In [ ]:
step = "block3c"

if not _is_done(step):
    import bisect

    LTAFDB_RESAMP_UP = 45
    LTAFDB_RESAMP_DOWN = 16
    LTAFDB_MAX_AFIB = 2000
    LTAFDB_MAX_PVC = 200

    ectopic_symbols = {
        "V", "r", "E", "F", "L", "R", "B", "!", "[", "]"
    }
    pvc_allowed = {"N", ".", "V"}

    rng_afib = np.random.default_rng(42)
    rng_pvc = random.Random(99)

    def choose_ltaf_pvc_position():
        if rng_pvc.random() < 0.40:
            return HALF_WIN
        return rng_pvc.randint(
            int(0.20 * WIN_LEN),
            int(0.80 * WIN_LEN),
        )

    if _has_hea_files(LTAFDB_LOCAL):
        ltafdb_records = sorted(
            os.path.splitext(file)[0]
            for file in os.listdir(LTAFDB_LOCAL)
            if file.endswith(".hea")
        )
    else:
        ltafdb_records = wfdb.get_record_list("ltafdb")

    totals = {"AFIB": 0, "PVC": 0}

    for record in ltafdb_records:
        record_path, pn_dir = _wfdb_args(
            LTAFDB_LOCAL,
            record,
            "ltafdb",
        )

        try:
            signal_data = wfdb.rdrecord(
                record_path,
                channels=[0],
                pn_dir=pn_dir,
            ).p_signal[:, 0].astype(np.float64)

            annotations = wfdb.rdann(
                record_path,
                "atr",
                pn_dir=pn_dir,
            )
        except Exception:
            continue

        signal_360 = sp_signal.resample_poly(
            signal_data,
            LTAFDB_RESAMP_UP,
            LTAFDB_RESAMP_DOWN,
        ).astype(np.float32)

        scale = LTAFDB_RESAMP_UP / LTAFDB_RESAMP_DOWN
        signal_length = len(signal_360)

        beats = [
            (int(sample * scale), symbol)
            for sample, symbol in zip(
                annotations.sample,
                annotations.symbol,
            )
        ]

        rhythm_events = []

        for sample, symbol, aux_note in zip(
            annotations.sample,
            annotations.symbol,
            annotations.aux_note,
        ):
            if symbol == "+":
                rhythm = aux_note.strip().rstrip("\x00").strip()
                rhythm_events.append((int(sample * scale), rhythm))

        if not rhythm_events:
            continue

        intervals = []

        for index, (start, rhythm) in enumerate(rhythm_events):
            end = (
                rhythm_events[index + 1][0]
                if index + 1 < len(rhythm_events)
                else signal_length
            )
            intervals.append((start, end, rhythm))

        interval_starts = [start for start, _, _ in intervals]
        afib_intervals = [
            (start, end)
            for start, end, rhythm in intervals
            if rhythm == "(AFIB"
        ]

        record_counts = {"AFIB": 0, "PVC": 0}

        afib_starts = []

        for start, end in afib_intervals:
            start = max(0, start)
            end = min(signal_length, end)

            for offset in range(start, end - WIN_LEN + 1, WIN_LEN):
                window_end = offset + WIN_LEN
                symbols = [
                    symbol
                    for sample, symbol in beats
                    if offset <= sample < window_end
                ]

                if not any(
                    symbol in ectopic_symbols for symbol in symbols
                ):
                    afib_starts.append(offset)

        if len(afib_starts) > LTAFDB_MAX_AFIB:
            afib_starts = list(
                rng_afib.choice(
                    afib_starts,
                    size=LTAFDB_MAX_AFIB,
                    replace=False,
                )
            )

        for offset in afib_starts:
            filename = (
                f"ltafdb_{record}_AFIB_"
                f"{record_counts['AFIB']:05d}.npy"
            )
            output_path = os.path.join(
                NSR_AFIB_DIR,
                "AFIB",
                filename,
            )

            np.save(output_path, signal_360[offset:offset + WIN_LEN])
            record_counts["AFIB"] += 1

        pvc_starts = []

        for sample, symbol in beats:
            if symbol != "V":
                continue

            interval_index = bisect.bisect_right(
                interval_starts,
                sample,
            ) - 1

            if interval_index < 0:
                continue

            if intervals[interval_index][2] == "(AFIB":
                continue

            target_position = choose_ltaf_pvc_position()
            start = sample - target_position
            end = start + WIN_LEN

            if start < 0 or end > signal_length:
                continue

            symbols = [
                beat_symbol
                for beat_sample, beat_symbol in beats
                if start <= beat_sample < end
            ]

            if not symbols or not all(
                beat_symbol in pvc_allowed
                for beat_symbol in symbols
            ):
                continue

            pvc_starts.append(start)

        if len(pvc_starts) > LTAFDB_MAX_PVC:
            pvc_starts = rng_pvc.sample(
                pvc_starts,
                LTAFDB_MAX_PVC,
            )

        for offset in pvc_starts:
            filename = (
                f"ltafdb_{record}_PVC_"
                f"{record_counts['PVC']:05d}.npy"
            )
            output_path = os.path.join(
                PVC_LBBB_DIR,
                "PVC",
                filename,
            )

            np.save(output_path, signal_360[offset:offset + WIN_LEN])
            record_counts["PVC"] += 1

        for label in totals:
            totals[label] += record_counts[label]

    _mark_done(step)

print("Block 3c complete.")

## Block 3d - Extract PVC and LBBB from INCART

INCART records are loaded from PhysioNet.

- Signals are resampled from 257 Hz to 360 Hz using `resample_poly(x, 360, 257)`.
- Lead II is used when available; otherwise, channel 0 is selected.
- `V` annotations are labeled as PVC.
- `L` annotations are labeled as LBBB.
- PVC windows use the Block 3 jitter policy:
  - 40% centered at sample 512
  - 60% uniformly positioned between samples 205 and 819
- LBBB windows are centered at sample 512.
- Windows may contain only normal beats or beats from the target class.
- Windows crossing the signal boundary are skipped.

Output directories:

- `pvc-lbbb/PVC/`
- `pvc-lbbb/LBBB/`

In [ ]:
step = "block3d"

if not _is_done(step):
    INCART_RESAMP_UP = 360
    INCART_RESAMP_DOWN = 257
    label_map = {"V": "PVC", "L": "LBBB"}
    allowed_beats = {
        "V": {"N", ".", "V"},
        "L": {"N", ".", "L"},
    }

    rng = random.Random(84)

    def choose_incart_position():
        if rng.random() < 0.40:
            return HALF_WIN
        return rng.randint(
            int(0.20 * WIN_LEN),
            int(0.80 * WIN_LEN),
        )

    def get_lead2_channel(signal_names):
        for index, name in enumerate(signal_names):
            if name.strip().upper() in {
                "II",
                "MLII",
                "ML II",
                "LEAD II",
            }:
                return index
        return 0

    if _has_hea_files(INCART_LOCAL):
        incart_records = sorted(
            os.path.splitext(file)[0]
            for file in os.listdir(INCART_LOCAL)
            if file.endswith(".hea")
        )
    else:
        incart_records = wfdb.get_record_list("incartdb")

    totals = {"PVC": 0, "LBBB": 0}

    for record in incart_records:
        record_path, pn_dir = _wfdb_args(
            INCART_LOCAL,
            record,
            "incartdb",
        )

        try:
            header = wfdb.rdheader(record_path, pn_dir=pn_dir)
            channel = get_lead2_channel(header.sig_name)

            signal_data = wfdb.rdrecord(
                record_path,
                channels=[channel],
                pn_dir=pn_dir,
            ).p_signal[:, 0].astype(np.float64)

            annotations = wfdb.rdann(
                record_path,
                "atr",
                pn_dir=pn_dir,
            )
        except Exception:
            continue

        signal_360 = sp_signal.resample_poly(
            signal_data,
            INCART_RESAMP_UP,
            INCART_RESAMP_DOWN,
        ).astype(np.float32)

        scale = INCART_RESAMP_UP / INCART_RESAMP_DOWN
        beats = [
            (int(sample * scale), symbol)
            for sample, symbol in zip(
                annotations.sample,
                annotations.symbol,
            )
        ]

        counts = {"PVC": 0, "LBBB": 0}

        for sample, symbol in beats:
            if symbol not in label_map:
                continue

            label = label_map[symbol]
            position = (
                choose_incart_position()
                if symbol == "V"
                else HALF_WIN
            )

            start = sample - position
            end = start + WIN_LEN

            if start < 0 or end > len(signal_360):
                continue

            symbols = [
                beat_symbol
                for beat_sample, beat_symbol in beats
                if start <= beat_sample < end
            ]

            if any(
                beat_symbol not in allowed_beats[symbol]
                for beat_symbol in symbols
            ):
                continue

            filename = f"incart_{record}_{label}_{counts[label]:05d}.npy"
            output_path = os.path.join(
                PVC_LBBB_DIR,
                label,
                filename,
            )

            np.save(output_path, signal_360[start:end])
            counts[label] += 1

        for label in totals:
            totals[label] += counts[label]

    _mark_done(step)

seal_step = "seal_segs"

if not _is_done(seal_step) and all(
    _is_done(step) for step in _EXTRACTION_STEPS
):
    _compress_to_drive(
        os.path.join(TMP_BASE, "segs"),
        SEGS_TAR_DRIVE,
        arc_name="segs",
    )
    _mark_done(seal_step)

print("Block 3d complete.")

## Block 4 - Patient-Wise Train, Validation, and Test Split

Segments are split at the record level so that all segments from one record remain in the same partition.

- Records are grouped by class signature and database.
- The target split is approximately:
  - 75% training
  - 12.5% validation
  - 12.5% testing
- Validation and test sets each contain at least one LBBB-only and AFIB-only record when available.
- A per-class segment cap limits the contribution of any single source.

Output files:

- `split/record_split.json`
- `split/train.csv`
- `split/val.csv`
- `split/test.csv`

In [ ]:
import random
import subprocess
import time

try:
    subprocess.run(
        ["fusermount", "-uz", "/content/drive"],
        capture_output=True,
    )
    time.sleep(3)

    from google.colab import drive

    drive.mount("/content/drive", force_remount=True)
except Exception:
    pass

os.makedirs(SPLIT_DIR, exist_ok=True)

folders = {
    "PVC": os.path.join(PVC_LBBB_DIR, "PVC"),
    "LBBB": os.path.join(PVC_LBBB_DIR, "LBBB"),
    "NSR": os.path.join(NSR_AFIB_DIR, "NSR"),
    "AFIB": os.path.join(NSR_AFIB_DIR, "AFIB"),
}


def get_record_id(filename):
    parts = os.path.splitext(filename)[0].split("_")

    if parts[0] == "ptbxl":
        return f"ptbxl_p{parts[1]}"
    if parts[0] == "ltafdb":
        return f"ltafdb_{parts[1]}"
    if parts[0] == "incart":
        return f"incart_{parts[1]}"

    return parts[0]


def get_database(record):
    if record.startswith("ptbxl_"):
        return "ptbxl"
    if record.startswith("ltafdb_"):
        return "ltafdb"
    if record.startswith("incart_"):
        return "incart"

    return "mitbih"


def distribute(records):
    records = sorted(records)
    n = len(records)

    if n == 1:
        return records, [], []
    if n == 2:
        return records[:1], records[1:], []
    if n == 3:
        return records[:1], records[1:2], records[2:]

    n_val = max(1, round(n * 0.125))
    n_test = max(1, round(n * 0.125))
    n_train = n - n_val - n_test

    return (
        records[:n_train],
        records[n_train:n_train + n_val],
        records[n_train + n_val:],
    )


record_files = defaultdict(lambda: defaultdict(list))

for label, folder in folders.items():
    for filename in os.listdir(folder):
        if not filename.endswith(".npy"):
            continue

        record = get_record_id(filename)
        record_files[record][label].append(
            os.path.join(folder, filename)
        )

records = sorted(record_files)
groups = defaultdict(list)

for record in records:
    signature = (
        get_database(record),
        frozenset(record_files[record]),
    )
    groups[signature].append(record)

record_split = {}

for group_records in groups.values():
    train, val, test = distribute(group_records)

    for record in train:
        record_split[record] = "train"
    for record in val:
        record_split[record] = "val"
    for record in test:
        record_split[record] = "test"

with open(os.path.join(SPLIT_DIR, "record_split.json"), "w") as f:
    json.dump(record_split, f, indent=2, sort_keys=True)

split_data = {
    "train": [],
    "val": [],
    "test": [],
}

for record in records:
    split = record_split[record]

    for label, paths in record_files[record].items():
        for path in paths:
            split_data[split].append((path, label, record))

max_per_class = {
    "train": 30_000,
    "val": 2_000,
    "test": 2_000,
}

random.seed(42)

for split, rows in split_data.items():
    grouped = defaultdict(list)

    for row in rows:
        grouped[row[1]].append(row)

    limited = []

    for label, label_rows in grouped.items():
        if len(label_rows) > max_per_class[split]:
            label_rows = random.sample(
                label_rows,
                max_per_class[split],
            )

        limited.extend(label_rows)

    split_data[split] = limited

    with open(
        os.path.join(SPLIT_DIR, f"{split}.csv"),
        "w",
        newline="",
    ) as f:
        writer = csv.writer(f)
        writer.writerow(["filepath", "class", "record"])
        writer.writerows(limited)

total_segments = sum(
    len(rows) for rows in split_data.values()
)

if total_segments == 0:
    raise RuntimeError("No segments found.")

print("Block 4 complete.")

## Block 5 - Add NSTDB Noise to the Training Set

Noise recordings are loaded from PhysioNet (`nstdb`).

- The training set receives an equal number of augmented samples per class.
- The number of samples is based on the total clean training segments and four noise variants.
- Each selected segment produces four versions:
  - Baseline wander (`BW`)
  - Electrode motion (`EM`)
  - Muscle artifact (`MA`)
  - Combined noise (`ALL`)
- Noise is added at a random SNR between 5 and 10 dB.
- Noise windows are sampled from the highest-RMS 70% of available windows.
- Validation and test CSV files are copied without changes.

Output:

- `split-augmented/augmented/`
- `split-augmented/train.csv`
- `split-augmented/val.csv`
- `split-augmented/test.csv`

In [ ]:
def load_noise_recording(record, pn_dir="nstdb", channel=0):
    return wfdb.rdrecord(record, pn_dir=pn_dir).p_signal[:, channel].astype(
        np.float64
    )


def build_active_windows(signal_data, window_length, percentile):
    n_windows = len(signal_data) // window_length
    rms = np.array([
        np.sqrt(np.mean(
            signal_data[i * window_length:(i + 1) * window_length] ** 2
        ))
        for i in range(n_windows)
    ])

    threshold = np.percentile(rms, percentile)

    return [
        i * window_length
        for i, value in enumerate(rms)
        if value > threshold
    ]


def sample_noise_window(signal_data, starts, window_length, rng):
    start = rng.choice(starts)
    return signal_data[start:start + window_length].copy()


def scale_noise_to_snr(clean, noise, snr_db):
    clean_rms = np.sqrt(np.mean(clean ** 2))
    noise_rms = np.sqrt(np.mean(noise ** 2))

    if clean_rms < 1e-12 or noise_rms < 1e-12:
        return noise

    target_rms = clean_rms / (10 ** (snr_db / 20))
    return noise * (target_rms / noise_rms)


def scale_combined_noise(clean, noises, snr_db):
    clean_rms = np.sqrt(np.mean(clean ** 2))

    if clean_rms < 1e-12:
        return noises

    target_rms = clean_rms / (10 ** (snr_db / 20))
    component_rms = target_rms / np.sqrt(len(noises))
    scaled = []

    for noise in noises:
        noise_rms = np.sqrt(np.mean(noise ** 2))

        if noise_rms < 1e-12:
            scaled.append(noise)
        else:
            scaled.append(noise * (component_rms / noise_rms))

    return scaled


step = "block5"

if not _is_done(step):
    os.makedirs(AUG_DIR, exist_ok=True)

    noise_types = {
        "BW": "bw",
        "EM": "em",
        "MA": "ma",
    }

    noise_signals = {}
    active_windows = {}

    for noise_type, record in noise_types.items():
        signal_data = load_noise_recording(record)
        noise_signals[noise_type] = signal_data
        active_windows[noise_type] = build_active_windows(
            signal_data,
            WIN_LEN,
            30,
        )

    if os.path.isdir(AUG_SEGS_DIR):
        shutil.rmtree(AUG_SEGS_DIR)

    os.makedirs(AUG_SEGS_DIR, exist_ok=True)

    for split in ("val", "test"):
        shutil.copy2(
            os.path.join(SPLIT_DIR, f"{split}.csv"),
            os.path.join(AUG_DIR, f"{split}.csv"),
        )

    with open(os.path.join(SPLIT_DIR, "train.csv"), newline="") as f:
        train_rows = list(csv.DictReader(f))

    rows_by_class = defaultdict(list)

    for index, row in enumerate(train_rows):
        rows_by_class[row["class"]].append(index)

    rng = random.Random(42)
    quota = len(train_rows) // (4 * len(rows_by_class))
    selected = []

    for label in sorted(rows_by_class):
        selected.extend(
            rng.sample(
                rows_by_class[label],
                min(quota, len(rows_by_class[label])),
            )
        )

    def augment_one(index):
        row = train_rows[index]
        clean = np.load(row["filepath"]).astype(np.float64)[:WIN_LEN]
        base_name = os.path.splitext(
            os.path.basename(row["filepath"])
        )[0]

        snr = rng.uniform(5.0, 10.0)
        noise = {
            noise_type: sample_noise_window(
                noise_signals[noise_type],
                active_windows[noise_type],
                WIN_LEN,
                rng,
            )
            for noise_type in noise_types
        }

        results = []

        for noise_type, noise_signal in noise.items():
            noisy = clean + scale_noise_to_snr(
                clean,
                noise_signal,
                snr,
            )

            filename = f"{base_name}_{noise_type}.npy"
            path = os.path.join(AUG_SEGS_DIR, filename)

            np.save(path, noisy.astype(np.float32))
            results.append({
                "filepath": path,
                "class": row["class"],
                "record": row["record"],
            })

        combined = sum(
            scale_combined_noise(
                clean,
                list(noise.values()),
                snr,
            )
        )

        filename = f"{base_name}_ALL.npy"
        path = os.path.join(AUG_SEGS_DIR, filename)

        np.save(path, (clean + combined).astype(np.float32))
        results.append({
            "filepath": path,
            "class": row["class"],
            "record": row["record"],
        })

        return index, results

    augmented = {}

    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = [executor.submit(augment_one, index) for index in selected]

        for future in as_completed(futures):
            index, rows = future.result()
            augmented[index] = rows

    new_rows = []

    for index in selected:
        new_rows.extend(augmented[index])

    output_rows = train_rows + new_rows
    train_path = os.path.join(AUG_DIR, "train.csv")

    with open(train_path, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["filepath", "class", "record"],
        )
        writer.writeheader()
        writer.writerows(output_rows)

    _compress_to_drive(
        AUG_SEGS_DIR,
        AUG_TAR_DRIVE,
        arc_name="aug_segs",
    )

    _mark_done(step)

print("Block 5 complete.")

## Block 6 - Filter and Normalize Segments

Each segment in the augmented dataset is processed as follows:

1. Remove the linear trend
2. Apply a 1 Hz high-pass Butterworth filter
3. Apply a 35 Hz low-pass Butterworth filter
4. Apply a 61.7 Hz notch filter with `Q=10`
5. Smooth with a Savitzky-Golay filter using:
   - Window length: 15
   - Polynomial order: 3
6. Apply z-score normalization

The IIR filters are combined into one SOS filter chain.

Output files:

- `split-filtered/train.npz`
- `split-filtered/val.npz`
- `split-filtered/test.npz`

In [ ]:
HPF_CUTOFF_HZ = 1.0
LPF_CUTOFF_HZ = 35.0
NOTCH_FREQ_HZ = 61.7
NOTCH_Q = 10.0
SG_WINDOW = 15
SG_POLYORDER = 3

sos = np.vstack([
    sp_signal.butter(
        4,
        HPF_CUTOFF_HZ,
        btype="high",
        fs=SAMPLE_RATE_HZ,
        output="sos",
    ),
    sp_signal.butter(
        4,
        LPF_CUTOFF_HZ,
        btype="low",
        fs=SAMPLE_RATE_HZ,
        output="sos",
    ),
    sp_signal.tf2sos(
        *sp_signal.iirnotch(
            NOTCH_FREQ_HZ,
            NOTCH_Q,
            SAMPLE_RATE_HZ,
        )
    ),
])


def filter_and_normalize(signal_data):
    signal_data = signal_data.astype(np.float64)
    signal_data = sp_signal.detrend(signal_data)
    signal_data = sp_signal.sosfiltfilt(sos, signal_data)
    signal_data = savgol_filter(
        signal_data,
        SG_WINDOW,
        SG_POLYORDER,
    )

    std = np.std(signal_data)

    std = max(std, 1e-8)

    return (
        (signal_data - np.mean(signal_data)) / std
    ).astype(np.float32)


def filter_row(row):
    try:
        signal_data = filter_and_normalize(
            np.load(row["filepath"])
        )

        if len(signal_data) < WIN_LEN:
            signal_data = np.pad(
                signal_data,
                (0, WIN_LEN - len(signal_data)),
            )
        elif len(signal_data) > WIN_LEN:
            signal_data = signal_data[:WIN_LEN]

        return signal_data, row["class"], None
    except Exception as error:
        return None, row["class"], error


def process_split(split):
    input_csv = os.path.join(AUG_DIR, f"{split}.csv")

    with open(input_csv, newline="") as f:
        rows = list(csv.DictReader(f))

    signals = [None] * len(rows)
    labels = np.full(len(rows), -1, dtype=np.int32)
    errors = 0

    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = {
            executor.submit(filter_row, row): index
            for index, row in enumerate(rows)
        }

        for future in as_completed(futures):
            index = futures[future]
            signal_data, label, error = future.result()

            if error is not None:
                errors += 1
                continue

            signals[index] = signal_data
            labels[index] = CLASS_TO_IDX[label]

    kept = [
        index
        for index, signal_data in enumerate(signals)
        if signal_data is not None
    ]

    if kept:
        X = np.stack([signals[index] for index in kept])
        y = labels[kept]
    else:
        X = np.empty((0, WIN_LEN), dtype=np.float32)
        y = np.empty(0, dtype=np.int32)

    filtered_rows = [rows[index] for index in kept]

    with open(
        os.path.join(FILTERED_DIR, f"{split}.csv"),
        "w",
        newline="",
    ) as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["filepath", "class", "record"],
        )
        writer.writeheader()
        writer.writerows(filtered_rows)

    if len(X) != len(y) or len(X) != len(filtered_rows):
        raise RuntimeError(f"{split} data alignment error")

    return X, y, errors


step = "block6"

if not _is_done(step):
    os.makedirs(FILTERED_DIR, exist_ok=True)

    total_saved = 0
    total_errors = 0

    for split in ("train", "val", "test"):
        X, y, errors = process_split(split)

        npz_path = os.path.join(FILTERED_DIR, f"{split}.npz")
        temp_path = f"/tmp/{split}.npz"

        np.savez_compressed(temp_path, X=X, y=y)
        shutil.copy2(temp_path, npz_path)
        os.remove(temp_path)

        total_saved += len(X)
        total_errors += errors

    _mark_done(step)

print("Block 6 complete.")

## Block 7 - Train and Evaluate the Dual-Input 1D CNN

The model has two inputs:

- An ECG signal branch using a residual 1D CNN
- A handcrafted feature branch containing RR-interval features

The signal branch includes:

- An initial convolution
- Three residual downsampling blocks
- A self-attention layer
- A morphology path using global max pooling
- A temporal path using a dilated convolution and global average pooling

The two signal paths are combined with the feature branch through a two-layer classification head.

### Training

- Sparse categorical cross-entropy loss
- Inverse-frequency class weights
- Adam optimizer
- Learning-rate reduction on plateau
- Early stopping based on validation macro-F1
- Best model weights restored after training
- Three runs with different random seeds

In [ ]:
BATCH_SIZE = 64
EPOCHS = 40
LEARNING_RATE = 5e-5

N_RUNS = 3
RUN_SEEDS = [42, 0, 7]

import csv
import datetime
import json

from scipy.signal import find_peaks, peak_widths


def extract_features(signals):
    features = []
    min_distance = int(0.2 * SAMPLE_RATE_HZ)

    for signal_data in signals:
        signal_data = signal_data.flatten()
        max_value = np.max(signal_data)

        if max_value < 0.1:
            features.append([0, 0, 0, 1.0, 0, 0, 0])
            continue

        peaks, _ = find_peaks(
            signal_data,
            height=max_value * 0.3,
            distance=min_distance,
        )

        if len(peaks) < 2:
            features.append([0, 0, 0, 1.0, 0, 0, 0])
            continue

        rr = np.diff(peaks) / SAMPLE_RATE_HZ
        rr_diff = np.diff(rr)

        mean_rr = np.mean(rr)
        std_rr = np.std(rr)
        rmssd = np.sqrt(np.mean(rr_diff ** 2)) if len(rr_diff) else 0
        pnn50 = (
            np.mean(np.abs(rr_diff) > 0.05)
            if len(rr_diff)
            else 0
        )

        widths, _, _, _ = peak_widths(
            signal_data,
            peaks,
            rel_height=0.5,
        )

        mean_qrs = np.mean(widths) / SAMPLE_RATE_HZ

        features.append([
            mean_rr,
            std_rr,
            rmssd,
            pnn50,
            np.min(rr),
            np.max(rr),
            mean_qrs,
        ])

    return np.asarray(features, dtype=np.float32)


def load_split(split):
    data = np.load(
        os.path.join(FILTERED_DIR, f"{split}.npz")
    )

    signals = data["X"].reshape(-1, WIN_LEN, 1)
    labels = data["y"]
    features = extract_features(signals)

    return signals, features, labels


X_train, X_train_feat, y_train = load_split("train")
X_val, X_val_feat, y_val = load_split("val")
X_test, X_test_feat, y_test = load_split("test")

class_ids, class_counts = np.unique(
    y_train,
    return_counts=True,
)

class_weight_dict = {
    class_id: len(y_train) / (len(CLASSES) * count)
    for class_id, count in zip(class_ids, class_counts)
}

class_weight_dict = {
    class_id: class_weight_dict.get(class_id, 1.0)
    for class_id in range(len(CLASSES))
}


def residual_block(x, filters, kernel_size, downsample=True):
    shortcut = x
    stride = 2 if downsample else 1

    if downsample or x.shape[-1] != filters:
        shortcut = layers.Conv1D(
            filters,
            1,
            strides=stride,
            padding="same",
        )(shortcut)

    regularizer = tf.keras.regularizers.l2(1e-4)

    y = layers.Conv1D(
        filters,
        kernel_size,
        strides=stride,
        padding="same",
        kernel_regularizer=regularizer,
    )(x)
    y = layers.BatchNormalization()(y)
    y = layers.Activation("relu")(y)

    y = layers.Conv1D(
        filters,
        kernel_size,
        padding="same",
        kernel_regularizer=regularizer,
    )(y)
    y = layers.BatchNormalization()(y)

    y = layers.Add()([shortcut, y])
    y = layers.Activation("relu")(y)
    y = layers.SpatialDropout1D(0.3)(y)

    return y


def build_dual_model(
    input_len=WIN_LEN,
    n_features=7,
    n_classes=4,
):
    signal_input = layers.Input(
        shape=(input_len, 1),
        name="signal_in",
    )

    x = layers.Conv1D(32, 7, padding="same")(signal_input)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = residual_block(x, 64, 31)
    x = residual_block(x, 128, 15)
    x = residual_block(x, 256, 5)

    attention = layers.Dense(1, activation="tanh")(x)
    attention = layers.Flatten()(attention)
    attention = layers.Activation("softmax")(attention)
    attention = layers.RepeatVector(256)(attention)
    attention = layers.Permute([2, 1])(attention)

    x = layers.Multiply()([x, attention])
    morphology = layers.GlobalMaxPooling1D()(x)

    temporal = layers.Conv1D(
        64,
        3,
        dilation_rate=4,
        padding="same",
    )(x)
    temporal = layers.BatchNormalization()(temporal)
    temporal = layers.Activation("relu")(temporal)

    temporal = layers.Conv1D(
        64,
        3,
        dilation_rate=8,
        padding="same",
    )(temporal)
    temporal = layers.BatchNormalization()(temporal)
    temporal = layers.Activation("relu")(temporal)

    temporal = layers.Conv1D(
        128,
        3,
        dilation_rate=16,
        padding="same",
    )(temporal)
    temporal = layers.BatchNormalization()(temporal)
    temporal = layers.Activation("relu")(temporal)

    temporal = layers.Conv1D(
        128,
        3,
        dilation_rate=32,
        padding="same",
    )(temporal)
    temporal = layers.BatchNormalization()(temporal)
    temporal = layers.Activation("relu")(temporal)
    temporal = layers.GlobalAveragePooling1D()(temporal)

    signal_features = layers.Concatenate()([
        morphology,
        temporal,
    ])
    signal_features = layers.Dropout(0.4)(signal_features)

    feature_input = layers.Input(
        shape=(n_features,),
        name="feat_in",
    )

    features = layers.BatchNormalization()(feature_input)
    features = layers.Dense(128, activation="relu")(features)
    features = layers.Dropout(0.3)(features)
    features = layers.Dense(64, activation="relu")(features)
    features = layers.Dropout(0.2)(features)
    features = layers.Dense(32, activation="relu")(features)

    combined = layers.Concatenate()([
        signal_features,
        features,
    ])

    regularizer = tf.keras.regularizers.l2(0.001)

    x = layers.Dense(
        256,
        activation="relu",
        kernel_regularizer=regularizer,
    )(combined)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizer,
    )(x)
    x = layers.Dropout(0.4)(x)

    output = layers.Dense(
        n_classes,
        activation="softmax",
    )(x)

    return models.Model(
        inputs=[signal_input, feature_input],
        outputs=output,
    )

In [ ]:
from sklearn.metrics import precision_recall_fscore_support


class MacroF1Callback(tf.keras.callbacks.Callback):
    def __init__(self, validation_data, labels):
        super().__init__()
        self.validation_data = validation_data
        self.labels = labels

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        predictions = np.argmax(
            self.model.predict(self.validation_data, verbose=0),
            axis=1,
        )
        score = f1_score(
            self.labels,
            predictions,
            average="macro",
        )
        logs["val_macro_f1"] = float(score)


def train_run(seed, run_index):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = os.path.join(
        RUNS_DIR,
        f"run_{run_index:02d}_seed{seed}_{timestamp}",
    )
    os.makedirs(run_dir, exist_ok=True)

    tf.keras.utils.set_random_seed(seed)

    model = build_dual_model(
        input_len=WIN_LEN,
        n_features=7,
        n_classes=len(CLASSES),
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE,
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    best_model = os.path.join(
        run_dir,
        "best_model_filtered.keras",
    )

    callbacks = [
        MacroF1Callback(
            [X_val, X_val_feat],
            y_val,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            best_model,
            monitor="val_macro_f1",
            mode="max",
            save_best_only=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_macro_f1",
            mode="max",
            factor=0.5,
            patience=5,
            min_delta=0.002,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_macro_f1",
            mode="max",
            patience=12,
            min_delta=0.002,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.CSVLogger(
            os.path.join(run_dir, "training_log.csv")
        ),
    ]

    history = model.fit(
        [X_train, X_train_feat],
        y_train,
        validation_data=([X_val, X_val_feat], y_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        class_weight=class_weight_dict,
        callbacks=callbacks,
        verbose=1,
    )

    model.save(os.path.join(run_dir, "final_model_dual.keras"))

    test_loss, test_accuracy = model.evaluate(
        [X_test, X_test_feat],
        y_test,
        verbose=0,
    )

    probabilities = model.predict(
        [X_test, X_test_feat],
        verbose=0,
    )
    predictions = np.argmax(probabilities, axis=1)

    report = classification_report(
        y_test,
        predictions,
        target_names=CLASSES,
        digits=4,
    )

    precision_macro = precision_score(
        y_test,
        predictions,
        average="macro",
    )
    recall_macro = recall_score(
        y_test,
        predictions,
        average="macro",
    )
    f1_macro = f1_score(
        y_test,
        predictions,
        average="macro",
    )

    precision_weighted = precision_score(
        y_test,
        predictions,
        average="weighted",
    )
    recall_weighted = recall_score(
        y_test,
        predictions,
        average="weighted",
    )
    f1_weighted = f1_score(
        y_test,
        predictions,
        average="weighted",
    )

    with open(
        os.path.join(run_dir, "classification_report.txt"),
        "w",
    ) as f:
        f.write(report)

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_test,
            predictions,
            labels=list(range(len(CLASSES))),
        )
    )

    with open(
        os.path.join(run_dir, "per_class_metrics.csv"),
        "w",
        newline="",
    ) as f:
        writer = csv.writer(f)
        writer.writerow([
            "class",
            "precision",
            "recall",
            "f1",
            "support",
        ])

        for index, label in enumerate(CLASSES):
            writer.writerow([
                label,
                precision[index],
                recall[index],
                f1[index],
                support[index],
            ])

    cm = confusion_matrix(y_test, predictions)

    with open(
        os.path.join(run_dir, "confusion_matrix.csv"),
        "w",
        newline="",
    ) as f:
        writer = csv.writer(f)
        writer.writerow(["actual/predicted"] + CLASSES)

        for index, row in enumerate(cm):
            writer.writerow([CLASSES[index]] + list(row))

    plt.figure(figsize=(7, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASSES,
        yticklabels=CLASSES,
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(
        os.path.join(run_dir, "confusion_matrix.png"),
        dpi=150,
    )
    plt.close()

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history["loss"])
    plt.plot(history.history["val_loss"])
    plt.title("Loss")
    plt.legend(["train", "validation"])

    plt.subplot(1, 2, 2)
    plt.plot(history.history["accuracy"])
    plt.plot(history.history["val_accuracy"])
    plt.title("Accuracy")
    plt.legend(["train", "validation"])

    plt.tight_layout()
    plt.savefig(
        os.path.join(run_dir, "learning_curves.png"),
        dpi=150,
    )
    plt.close()

    summary = {
        "run_index": run_index,
        "seed": seed,
        "run_dir": run_dir,
        "stopped_epoch": len(history.history["loss"]),
        "best_val_loss": float(min(history.history["val_loss"])),
        "best_val_accuracy": float(max(history.history["val_accuracy"])),
        "best_val_macro_f1": float(max(history.history["val_macro_f1"])),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_accuracy),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "f1_macro": float(f1_macro),
        "precision_weighted": float(precision_weighted),
        "recall_weighted": float(recall_weighted),
        "f1_weighted": float(f1_weighted),
    }

    with open(
        os.path.join(run_dir, "run_meta.json"),
        "w",
    ) as f:
        json.dump(summary, f, indent=2)

    tf.keras.backend.clear_session()
    return summary


os.makedirs(RUNS_DIR, exist_ok=True)

run_summaries = [
    train_run(seed, index)
    for index, seed in enumerate(RUN_SEEDS)
]

summary_path = os.path.join(
    RUNS_DIR,
    f"multi_run_summary_"
    f"{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.csv",
)

with open(summary_path, "w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=run_summaries[0].keys(),
    )
    writer.writeheader()
    writer.writerows(run_summaries)

print("Block 7 complete.")

## Block 8 - TFLite Conversion

Convert the best model from each training run into two TFLite formats:

1. Standard float32 model
2. Dynamic-range quantized model

The converted models are saved in their respective run directories:

```
run_XX_seedY_<ts>/
├── best_model.tflite
└── best_model_quant.tflite
```

In [ ]:
_tflite_rows = []

for _s in all_run_summaries:
    _run_dir   = _s['run_dir']
    _run_label = os.path.basename(_run_dir)
    _keras_path = os.path.join(_run_dir, 'best_model_filtered.keras')
    _std_path   = os.path.join(_run_dir, 'best_model.tflite')
    _quant_path = os.path.join(_run_dir, 'best_model_quant.tflite')


    _model = tf.keras.models.load_model(_keras_path, compile=False)

    _conv = tf.lite.TFLiteConverter.from_keras_model(_model)
    _std_buf = _conv.convert()
    with open(_std_path, 'wb') as _f:
        _f.write(_std_buf)

    _conv = tf.lite.TFLiteConverter.from_keras_model(_model)
    _conv.optimizations = [tf.lite.Optimize.DEFAULT]
    _quant_buf = _conv.convert()
    with open(_quant_path, 'wb') as _f:
        _f.write(_quant_buf)

    _kb = lambda p: os.path.getsize(p) / 1024
    _keras_kb = _kb(_keras_path)
    _std_kb   = _kb(_std_path)
    _quant_kb = _kb(_quant_path)

    _tflite_rows.append({
        'run':        _run_label,
        'keras_kb':   round(_keras_kb, 1),
        'std_kb':     round(_std_kb,   1),
        'quant_kb':   round(_quant_kb, 1),
    })

    tf.keras.backend.clear_session()

print('Block 8 complete.')